<a href="https://colab.research.google.com/github/RanitadelEstanque/Test-Liverpool/blob/main/Test_Liverpool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ==============================================================================
# 1. CONECTAR GOOGLE COLAB CON TU GOOGLE SHEETS
# ==============================================================================
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

creds, _ = default()
gc = gspread.authorize(creds)

# ⚠️ REEMPLAZA AQUÍ: Pon el nombre exacto de tu archivo de Google Sheets
nombre_del_archivo = "Test Liverpool_Allscenario"
spreadsheet = gc.open(nombre_del_archivo)

# ==============================================================================
# 2. LEER Y SANITIZAR DICCIONARIO DE MAPEO
# ==============================================================================
worksheet_mapeo = spreadsheet.worksheet("MAPEO_MODELOS")
df_mapeo = pd.DataFrame(worksheet_mapeo.get_all_records())

# Pasamos a minúsculas Y BORRAMOS ESPACIOS EXTRAS al principio o final (.str.strip())
df_mapeo['Palabra_Clave'] = df_mapeo['Palabra_Clave'].astype(str).str.lower().str.strip()
df_mapeo['Nombre_Comercial_Limpio'] = df_mapeo['Nombre_Comercial_Limpio'].astype(str).str.strip()

# Ordenamos por longitud para respetar jerarquías
df_mapeo['longitud'] = df_mapeo['Palabra_Clave'].str.len()
df_mapeo = df_mapeo.sort_values(by='longitud', ascending=False)

# Creamos el diccionario final libre de espacios fantasmas
diccionario_modelos = dict(zip(df_mapeo['Palabra_Clave'], df_mapeo['Nombre_Comercial_Limpio']))

# ==============================================================================
# 3. LEER DATA CRUDA
# ==============================================================================
worksheet_raw = spreadsheet.worksheet("RAW_Earbuds")
df = pd.DataFrame(worksheet_raw.get_all_records())

# ==============================================================================
# 4. FUNCIÓN MEJORADA CON DIAGNÓSTICO
# ==============================================================================
def clasificar_modelo_automatico(titulo):
    titulo_minusc = str(titulo).lower().strip()

    for palabra_clave, nombre_limpio in diccionario_modelos.items():
        if palabra_clave in titulo_minusc:
            return nombre_limpio

    # MEJORA: Si no lo encuentra, te muestra qué texto venía en "Producto Clean"
    return f"Revisar: {titulo}"

df['Producto_Final_BI'] = df['Producto Clean'].apply(clasificar_modelo_automatico)

# ==============================================================================
# 5. REPARAR LOS PRECIOS
# ==============================================================================
def corregir_precio(precio_sucio):
    precio_str = str(precio_sucio).replace('$', '').replace(',', '').replace('.', '').strip()
    if not precio_str or precio_str == 'none' or precio_str == '':
        return 0.0
    try:
        return float(precio_str) / 100
    except:
        return 0.0

df['Precio_Nuevo_Final'] = df['New'].apply(corregir_precio)
df['Precio_Histo_Final'] = df['Histo'].apply(corregir_precio)

# ==============================================================================
# 6. ENVIAR A MASTER_EARBUDS
# ==============================================================================
columnas_ejecutivas = {
    'Brand': 'Brand',
    'Producto_Final_BI': 'Product_Model',
    'Precio_Nuevo_Final': 'Current_Price',
    'Precio_Histo_Final': 'Historical_Price',
    'Link': 'URL_Source'
}

df_master = df[list(columnas_ejecutivas.keys())].rename(columns=columnas_ejecutivas)
df_master['Report_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
df_master['Channel'] = 'Liverpool'

try:
    worksheet_master = spreadsheet.worksheet("MASTER_Earbuds")
except:
    worksheet_master = spreadsheet.add_worksheet(title="MASTER_Earbuds", rows="1000", cols="10")

worksheet_master.clear()
worksheet_master.update([df_master.columns.values.tolist()] + df_master.values.tolist())

print("Done! Corrección aplicada.")

Done! Corrección aplicada.


### Deduplicate Products and Highlight Cheaper Prices

This section addresses the requirement to deduplicate products. It ensures that for any given `Product_Model`, only one entry is retained: the one with the lowest `Current_Price`. A new column, `Is_Cheapest_Among_Duplicates`, will be added to indicate if a product was selected as the cheapest among its duplicates.

In [13]:
# 1. Drop exact duplicates across all columns first (if any)
df_master_deduplicated = df_master.drop_duplicates().copy()

# 2. Sort by Product_Model and then by Current_Price (ascending)
# This prepares the DataFrame so that when duplicates are dropped, the cheapest one is kept.
df_master_deduplicated = df_master_deduplicated.sort_values(by=['Product_Model', 'Current_Price'], ascending=[True, True])

# 3. Add a column to indicate if it's the cheapest among its Product_Model duplicates
# Initialize the column to False
df_master_deduplicated['Is_Cheapest_Among_Duplicates'] = False

# Find the indices of the rows that have the minimum 'Current_Price' for each 'Product_Model' group
cheapest_indices = df_master_deduplicated.groupby('Product_Model')['Current_Price'].idxmin()

# Set 'Is_Cheapest_Among_Duplicates' to True for these rows
df_master_deduplicated.loc[cheapest_indices, 'Is_Cheapest_Among_Duplicates'] = True

# 4. Drop duplicates based on 'Product_Model', keeping the first occurrence
# (which is the one with the lowest price due to previous sorting)
df_master_deduplicated = df_master_deduplicated.drop_duplicates(subset='Product_Model', keep='first').copy()

print(f"Original number of rows: {len(df_master)}")
print(f"Number of rows after deduplication (keeping cheapest): {len(df_master_deduplicated)}")

display(df_master_deduplicated.head())

Original number of rows: 33
Number of rows after deduplication (keeping cheapest): 26


,Brand,Product_Model,Current_Price,Historical_Price,URL_Source,Report_Date,Channel,Is_Cheapest_Among_Duplicates
14,APPLE,AirPods 4,2999.0,0.0,https://www.liverpool.com.mx/tienda/pdp/apple-...,2026-06-10,Liverpool,True
23,HUAWEI,FreeBuds 7i,1799.0,0.0,https://www.liverpool.com.mx/tienda/pdp/aud%C3...,2026-06-10,Liverpool,True
31,HUAWEI,FreeBuds SE 4 ANC,1499.0,0.0,https://www.liverpool.com.mx/tienda/pdp/aud%C3...,2026-06-10,Liverpool,True
10,HUAWEI,FreeClip 2,3999.0,0.0,https://www.liverpool.com.mx/tienda/pdp/aud%C3...,2026-06-10,Liverpool,True
26,HUAWEI,Freebuds SE 3,999.0,0.0,https://www.liverpool.com.mx/tienda/pdp/aud%C3...,2026-06-10,Liverpool,True


### Export Deduplicated Data to Google Sheet

This section exports the `df_master_deduplicated` DataFrame, which contains unique product models with the lowest price, to a new worksheet in your Google Sheet named `MASTER_Earbuds_Deduplicated`. If the sheet doesn't exist, it will be created.

In [14]:
# Try to get the worksheet. If it doesn't exist, create it.
try:
    worksheet_deduplicated = spreadsheet.worksheet("MASTER_Earbuds_Deduplicated")
except gspread.exceptions.WorksheetNotFound:
    print("Creating new worksheet 'MASTER_Earbuds_Deduplicated'...")
    worksheet_deduplicated = spreadsheet.add_worksheet(title="MASTER_Earbuds_Deduplicated", rows="1000", cols="10")

# Clear existing content in the sheet
worksheet_deduplicated.clear()

# Update the sheet with the DataFrame's headers and data
worksheet_deduplicated.update([df_master_deduplicated.columns.values.tolist()] + df_master_deduplicated.values.tolist())

print("Successfully exported deduplicated data to 'MASTER_Earbuds_Deduplicated' worksheet.")

Successfully exported deduplicated data to 'MASTER_Earbuds_Deduplicated' worksheet.
